# Group Relative Policy Optimization (GRPO) with veRL on Amazon SageMaker Training jobs
## Lab 1 - Data preparation
In this notebook, we are going to prepare the [GSM8K](https://huggingface.co/datasets/openai/gsm8k) dataset in the schema that [veRL](https://github.com/volcengine/verl) expects for reinforcement learning, and upload it to Amazon S3.

## Prerequisites

### Install requirements

In [ ]:
%pip install -r ./requirements.txt --upgrade

### Setup and dependencies

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()

sagemaker_session_bucket = None
if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")

sess = Session(default_bucket=sagemaker_session_bucket)

bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

***

## Visualize the dataset
In this example, we are going to load [openai/gsm8k](https://huggingface.co/datasets/openai/gsm8k), 8.5k grade-school maths word problems. Each problem has a worked solution that ends with the final answer after a `####` marker.

That marker is the reason this dataset suits GRPO. Supervised fine-tuning and DPO both learn from text we hand them -- a target completion, or a preferred and a rejected one. GRPO learns from text the model writes itself: for every prompt it samples a group of completions, scores each one, and pushes the policy toward the ones that scored well. So the dataset does not need a good answer to imitate. It needs a **checkable** answer to score against.

In [ ]:
import datasets
from datasets import load_dataset

gsm8k = load_dataset("openai/gsm8k", "main")
gsm8k

In [ ]:
import pandas as pd

df = pd.DataFrame(gsm8k["train"])
df.head()

A single problem in full, so the `####` convention is visible.

In [ ]:
print(df.loc[0, "question"])
print()
print(df.loc[0, "answer"])

## The veRL reinforcement-learning schema
veRL reads its data as parquet with five columns, and each one has a job:

| column | what it carries |
| --- | --- |
| `data_source` | Which reward function scores this row |
| `prompt` | The chat messages the model is asked to complete |
| `ability` | A free-text tag, for filtering and reporting |
| `reward_model` | The scoring style, and the ground truth to score against |
| `extra_info` | Split name, row index, and anything else we want to keep |

`data_source` is the field worth pausing on, because it is doing something less obvious than it looks. We are not writing a reward function in this lab and we are not passing a flag to select one. veRL's default reward manager reads `data_source` off each row and dispatches on its value, so the literal string `openai/gsm8k` is what routes every row to veRL's built-in GSM8K scorer at `verl/utils/reward_score/gsm8k.py`. That scorer greps the model's completion for `#### <number>` and compares it to `reward_model.ground_truth`. Change the string and scoring silently stops working, so we set it once, in one place, below.

The second thing that scorer implies is a prompt requirement. It can only find an answer the model actually formats with `####`, so we append an instruction telling the model to do that. Without it the completions may well be correct and will still score zero.

Utility function to pull the final answer out of a solution.

In [ ]:
import re

DATA_SOURCE = "openai/gsm8k"
INSTRUCTION = 'Let\'s think step by step and output the final answer after "####".'


def extract_ground_truth(solution: str) -> str:
    """Pull the final answer out of a GSM8K solution.

    The solutions end with `#### <number>`. We keep the number only, with commas
    removed, because that is the form veRL's scorer compares against.
    """
    match = re.search(r"#### (\-?[0-9\.\,]+)", solution)
    if match is None:
        raise ValueError(f"no #### answer found in: {solution!r}")
    return match.group(0).split("#### ")[1].replace(",", "")


print(extract_ground_truth(df.loc[0, "answer"]))

Utility functions to map a GSM8K record into a veRL row, and to build a split.

In [ ]:
def to_verl_row(question: str, answer: str, split: str, index: int) -> dict:
    return {
        "data_source": DATA_SOURCE,
        "prompt": [{"role": "user", "content": f"{question} {INSTRUCTION}"}],
        "ability": "math",
        "reward_model": {
            "style": "rule",
            "ground_truth": extract_ground_truth(answer),
        },
        "extra_info": {
            "split": split,
            "index": index,
            "answer": answer,
            "question": question,
        },
    }


def build_split(hf_split, split_name: str, rows: int | None, seed: int = 42):
    prepared = [
        to_verl_row(record["question"], record["answer"], split_name, index)
        for index, record in enumerate(hf_split)
    ]
    prepared_ds = datasets.Dataset.from_list(prepared)
    if rows is not None and rows < len(prepared_ds):
        prepared_ds = prepared_ds.shuffle(seed=seed).select(range(rows))
    return prepared_ds

We are deliberately not training on all 7473 rows. GRPO samples `rollout_n` completions for every prompt in a step, so a step is far more expensive than a supervised one -- on `ml.g6e.12xlarge` it takes about ten minutes. 1280 rows at a batch size of 128 is exactly 10 steps, which is enough to move the policy measurably and short enough to finish inside a lab.

Validation is trimmed for the same reason: veRL generates a completion for every validation row, and it does so twice, once before training starts and once at the end. Those two passes are what let us say whether the ten steps changed anything, so we keep them cheap.

In [ ]:
# GSM8K ships train and test; veRL validates on what we mount as the
# validation channel, so the test split becomes validation here.
train_dataset = build_split(gsm8k["train"], "train", rows=1280)
validation_dataset = build_split(gsm8k["test"], "test", rows=256)

print(f"train rows:      {len(train_dataset)}")
print(f"validation rows: {len(validation_dataset)}")

Inspect one prepared row, which is what veRL will actually read.

In [ ]:
import random
from rich.pretty import pprint

pprint(train_dataset[random.randint(0, len(train_dataset) - 1)])

### Upload to Amazon S3
veRL loads a *file*, not a directory, so each channel path names the parquet file directly. We also keep a copy of the validation questions on local disk, which Lab 4 reads to evaluate the trained model.

In [ ]:
import os
import shutil

os.makedirs("./data/train", exist_ok=True)
os.makedirs("./data/validation", exist_ok=True)
os.makedirs("./tmp", exist_ok=True)

train_dataset.to_parquet("./data/train/dataset.parquet")
validation_dataset.to_parquet("./data/validation/dataset.parquet")

if default_prefix:
    input_path = f"{default_prefix}/datasets/llm-fine-tuning-grpo"
else:
    input_path = "datasets/llm-fine-tuning-grpo"

s3_client.upload_file(
    "./data/train/dataset.parquet", bucket_name, f"{input_path}/train/dataset.parquet"
)
s3_client.upload_file(
    "./data/validation/dataset.parquet",
    bucket_name,
    f"{input_path}/validation/dataset.parquet",
)

# Lab 4 evaluates against the same questions veRL validated on.
validation_dataset.to_json("./tmp/gsm8k_eval.jsonl")

shutil.rmtree("./data")

train_dataset_s3_path = f"s3://{bucket_name}/{input_path}/train/dataset.parquet"
validation_dataset_s3_path = (
    f"s3://{bucket_name}/{input_path}/validation/dataset.parquet"
)

print(f"train: {train_dataset_s3_path}")
print(f"validation: {validation_dataset_s3_path}")